# Numerics for HPC · what floats do when you aren't looking

This is **labCC**, an on-ramp on numerical hygiene. It is **optional but strongly recommended** — every parallelization lab from lab 03 onward can bite you with one of the pitfalls in here, and "my parallel version gives different answers!" is the number one panic-ticket in HPC courses. This lab is where you learn why, and what to do about it.

You already know from lab 01 that floating-point on a computer is not the reals. labCC digs into the specific ways that gap can bite an HPC program.

**You will build tiny C programs on the Hub** (no cluster round-trips — the whole lab runs in this Jupyter kernel) that demonstrate each pitfall on your own hardware:

1. Why `float` and `double` are actually different — bit-level view.
2. Why `a == b` is a bug for floating-point numbers, and what to write instead.
3. Catastrophic cancellation — how the textbook quadratic formula loses 12 digits.
4. `-ffast-math` and friends — what the compiler is allowed to change about your math.
5. Timer resolution — `time()`, `clock()`, `clock_gettime`, and which to use.
6. Reproducibility across N — why parallel reductions give slightly different answers, and when that matters.

**Every lab you do that touches new math or a new numerical trick will re-derive the relevant part in context**, so you never have to context-switch mid-lab. This lab is the one place all of it is collected — bookmark it and come back to Part N when Part N bites you.

> **📚 Where to look when you're stuck**
> 
> - [**IEEE 754 (Wikipedia)**](https://en.wikipedia.org/wiki/IEEE_754) — the standard > that defines what `float` and `double` are and how their bits are arranged.
> - [**What Every Computer Scientist Should Know About Floating-Point Arithmetic**](https://docs.oracle.com/cd/E19957-01/806-3568/ncg_goldberg.html) — Goldberg 1991, the canonical reference. Long, worth it.
> - [**The 0.30000000000000004 site**](https://0.30000000000000004.com/) — one > punchline, one page.
> - [**Numerical Recipes in C**](http://numerical.recipes/) — old but still gold on > practical numerical algorithms.


## How this notebook works · Hub-only lab, no cluster

Unlike lab 00, 01, and everything from lab 02 on, labCC **does not touch Crux**. Every cell here writes a tiny C program to your Hub filesystem, compiles it with your local `gcc`, and runs it in this kernel. That is deliberate — you should be able to review Part 3 or Part 5 while offline, with no MobilePASS+ passcode, no queue, no scratch.

| Where | How it looks | What it can do |
|---|---|---|
| **Hub** (this Jupyter kernel) | plain Python, or `!command` shell magic | write + compile + run C locally |


In [ ]:
# [Hub] Shared toolkit - same import as every other lab.
from labHelpers import *


### Set up this lab's identity

`setupLab` here writes a `labEnv.sh` for consistency with the other labs but does not target a cluster — there is nothing to preflight against Crux for this one.


In [ ]:
# [Hub] Local-only lab; no cluster ssh needed. remoteUser/project defaults are fine.
env = setupLab(labName="labCC", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               scratch="/tmp/labCCUnused")
labDir = pathlib.Path(env['labDir'])


### Preflight · you only need a local C compiler

No cluster checks — labCC only needs `cc` (or `gcc`) on the Hub. Every JupyterHub in the class ships with GCC installed by default.


In [ ]:
# [Hub] Local C toolchain must be present.
preflight([
    check("cc or gcc on this Hub", commandOnPath("cc"),
          hint="Ask on the class Slack. Every Hub image should have a C compiler pre-installed."),
], infoRows=[('host', 'Hub (local)'), ('lab dir', str(labDir))])


## Part 1 · What `float` and `double` really are

An [IEEE 754](https://en.wikipedia.org/wiki/IEEE_754) `float` is 32 bits: 1 sign bit, 8 exponent bits, 23 mantissa bits. A `double` is 64 bits: 1 sign, 11 exponent, 52 mantissa. The mantissa is what controls **how many decimal digits you can trust**.

- `float` mantissa (23 bits + implicit leading 1) → about **7 decimal digits** of precision.
- `double` mantissa (52 bits + implicit leading 1) → about **15-16 decimal digits**.

**Rule of thumb for HPC:**
- Use `double` for anything scientific unless you have a specific reason not to.
- `float` is worth considering for **memory-bound** kernels (twice the values in cache, twice the effective bandwidth) and for **AI/ML** (weights survive 7 digits of noise). Not for accumulators, iterative solvers, or long-time integration.
- Never mix — a `float * double` silently promotes the `float` to `double`; a `float + float` stays `float`. Grep your inner loops for stray literals like `2.0f` when you meant `2.0`.


In [ ]:
# [Hub] Write part1FloatVsDouble.c, compile it, run it.
srcPath = labDir / 'part1FloatVsDouble.c'
srcPath.write_text('/* part1FloatVsDouble.c\n * How many bits IEEE 754 gives you, and what that means in decimal.\n * The same "0.1" is a DIFFERENT number at 32-bit vs 64-bit precision.\n * Compile: cc -O2 -Wall -o part1 part1FloatVsDouble.c\n */\n#include <stdio.h>\n#include <string.h>\nint main(void) {\n    float  f = 0.1f;\n    double d = 0.1;\n    unsigned int  fBits;  memcpy(&fBits, &f, 4);\n    unsigned long dBits;  memcpy(&dBits, &d, 8);\n    printf("float  0.1f = %.20f  (hex bits: 0x%08x)\\n",  (double)f, fBits);\n    printf("double 0.1  = %.20f  (hex bits: 0x%016lx)\\n", d,        dBits);\n    printf("float  gives ~7 decimal digits of precision;\\n");\n    printf("double gives ~15-16 decimal digits of precision.\\n");\n    printf("Difference of the same decimal at the two precisions: %.3e\\n", (double)f - d);\n    return 0;\n}\n')
showFile(srcPath, language='c', maxLines=40, title='part1FloatVsDouble.c')


In [ ]:
# [Hub] Compile and run.
buildOut, buildRc = runShell(f"cd {labDir} && cc -O2 -Wall -o part1 part1FloatVsDouble.c")
if buildRc != 0:
    print('build failed:'); print(buildOut)
else:
    runOut, _ = runShell(f"{labDir}/part1")
    print(runOut)


### 🖊️ What did you just see?

Read the output above and answer these for yourself:

1. The last few digits of `float 0.1f` and `double 0.1` are different. **Which is closer to the mathematical value 1/10?**
2. The difference between the two is on the order of 1e-8. If you accumulate a `float` value 100 million times, roughly how far off will you be from the mathematical answer?
3. When would you deliberately choose `float` over `double`?


In [ ]:
checkpoint("Part 1 - built and ran the float-vs-double demo", [
    check("part1 binary built", fileExists(str(labDir / 'part1'))),
])


## Part 2 · Never `a == b` for floats

`0.1 + 0.2 == 0.3` is `false` in every IEEE 754 language, because none of the three decimals is exactly representable in binary. Sum the binary approximations, and you get a binary approximation to `0.3` that is off by one bit. See [the 0.30000000000000004 site](https://0.30000000000000004.com/) for a hall of shame.

The fix is to compare in a **tolerance**. For scientific data, an absolute tolerance (`fabs(a - b) <= 1e-10`) is usually what you want. For values that span many orders of magnitude, a relative tolerance (`fabs(a - b) / fabs(b) <= 1e-10`) is more robust. The deepest way is **ULP distance** — how many *representable* doubles apart the two numbers are.


In [ ]:
# [Hub] Write, compile, run part2.
(labDir / 'part2UlpAndEquality.c').write_text('/* part2UlpAndEquality.c\n * The classic 0.1 + 0.2 == 0.3 puzzler, and the ULP fix.\n * Compile: cc -O2 -Wall -o part2 part2UlpAndEquality.c\n */\n#include <stdio.h>\n#include <math.h>\n#include <string.h>\n\n/* ULP distance between two doubles as a non-negative integer. */\nstatic long ulpDist(double a, double b) {\n    long ia, ib;\n    memcpy(&ia, &a, 8);\n    memcpy(&ib, &b, 8);\n    if ((ia < 0) != (ib < 0)) return -1;   /* different signs, defined only around 0 */\n    long d = ia - ib;\n    return d < 0 ? -d : d;\n}\n\nint main(void) {\n    double a = 0.1 + 0.2;\n    double b = 0.3;\n    printf("0.1 + 0.2 = %.20f\\n", a);\n    printf("0.3       = %.20f\\n", b);\n    printf("a == b ?    %s   (this is why you never compare floats with ==)\\n",\n           a == b ? "yes" : "no");\n    printf("ULP distance:      %ld  (a and b are this many representable doubles apart)\\n",\n           ulpDist(a, b));\n    printf("Relative diff:     %.3e\\n", fabs(a - b) / fabs(b));\n    printf("\\nUse:  fabs(a - b) <= tol   (with tol chosen for your problem).\\n");\n    return 0;\n}\n')
buildOut, buildRc = runShell(f"cd {labDir} && cc -O2 -Wall -o part2 part2UlpAndEquality.c")
if buildRc != 0: print(buildOut)
else:
    print(runShell(f'{labDir}/part2')[0])


### 🖊️ Where this bites in HPC

1. Every parallel reduction (`MPI_Reduce`, `#pragma omp reduction`) can differ from the serial version in its last few bits. **When you write a regression test, use tolerances**, not `assert(result == expected)`.
2. "Bit-exact reproducibility" is not free. It requires fixing the reduction order — see Part 6.


In [ ]:
checkpoint("Part 2 - ULPs and equality", [
    check("part2 binary built", fileExists(str(labDir / 'part2'))),
])


## Part 3 · Catastrophic cancellation

When you subtract two nearly-equal floating-point numbers, the leading digits agree and cancel, and what you're left with is a small difference computed from the noise in the last few digits of each operand. You can lose 10+ digits of precision in a single subtraction. That is **catastrophic cancellation**.

The classic worked example is the quadratic formula. For $ax^2 + bx + c = 0$ with $a = 1$, $b = -200$, $c = 1.5 \times 10^{-5}$, one root is huge (~200) and one is tiny (~$7.5 \times 10^{-8}$). Computed the naive way, the tiny root subtracts two numbers that agree to 12 digits — so the answer has ~3 digits of precision, not 15.

The fix uses [**Vieta's formulas**](https://en.wikipedia.org/wiki/Vieta%27s_formulas) — compute the well-conditioned root first, then get the other from $x_1 x_2 = c/a$. No dangerous subtraction, full precision on both.


In [ ]:
# [Hub] Naive quadratic formula vs numerically stable form.
(labDir / 'part3Cancellation.c').write_text('/* part3Cancellation.c\n * Catastrophic cancellation: subtracting two nearly-equal numbers loses\n * almost all your precision. Solve x^2 - 200x + 0.000015 = 0 both the\n * textbook way and the numerically stable way. Compare answers.\n * Compile: cc -O2 -Wall -o part3 part3Cancellation.c\n */\n#include <stdio.h>\n#include <math.h>\n\nint main(void) {\n    double a = 1.0, b = -200.0, c = 0.000015;\n    double disc = sqrt(b*b - 4.0*a*c);\n\n    /* Textbook quadratic formula. Both roots. */\n    double x1_naive = (-b - disc) / (2.0 * a);   /* -(-200) - ~200 => cancels! */\n    double x2_naive = (-b + disc) / (2.0 * a);\n\n    /* Numerically stable form using Vieta: x1 * x2 = c/a.\n     * Compute the well-conditioned root first, then the other via c/(a*x). */\n    double xBig  = (-b + disc) / (2.0 * a);\n    double xSmall = c / (a * xBig);\n\n    printf("Naive:   x1 = %.15e   x2 = %.15e\\n", x1_naive, x2_naive);\n    printf("Stable:  x1 = %.15e   x2 = %.15e\\n", xSmall,   xBig);\n    printf("Small root, naive vs stable:  %+.3e absolute error\\n",\n           x1_naive - xSmall);\n    printf("\\nThe subtraction (-b - disc) with b nearly equal to -disc kills 12+ digits.\\n");\n    printf("Rule of thumb: subtract two nearly-equal numbers, expect to lose precision.\\n");\n    return 0;\n}\n')
buildOut, buildRc = runShell(f'cd {labDir} && cc -O2 -Wall -o part3 part3Cancellation.c')
if buildRc != 0: print(buildOut)
else: print(runShell(f'{labDir}/part3')[0])


### Where this bites in HPC

- **Long accumulators** — summing millions of small values into a single running total. Late in the sum, the running total dwarfs each new term, and each addition throws away the term's low bits. Fix: **Kahan summation** (an extra accumulator that tracks the lost bits). Every HPC library that offers a `sum` has this built in.
- **Finite differences of nearly-equal function values** — the very thing our lab 01 stencil does! When two neighbors are almost equal, `u[i+1] - u[i]` loses precision. In practice this is fine for well-conditioned diffusion because the ratios stay bounded, but it matters for wave problems and PDEs with shocks.
- **Distance between two points near the origin** in 3-D geometry code.


In [ ]:
checkpoint("Part 3 - catastrophic cancellation", [
    check("part3 binary built", fileExists(str(labDir / 'part3'))),
])


## Part 4 · `-ffast-math` and what the compiler is allowed to change

By default, a C compiler will not reorder your floating-point operations. Even if it can prove that `(a + b) + c` and `a + (b + c)` mathematically agree, it knows they don't under IEEE 754, and it leaves your code alone. `-ffast-math` tells the compiler: **I promise my program does not depend on IEEE semantics.** In exchange, the compiler reassociates freely, uses reciprocal approximations, and can vectorize reductions.

The speedup can be 2x on stencil code. The cost is that the answer can change, sometimes in the last bit, sometimes catastrophically (NaN handling changes, denormals get flushed to zero, infinity handling changes). **`-ffast-math` is not appropriate for scientific code that you're going to publish.**

See GCC's [Optimize Options](https://gcc.gnu.org/onlinedocs/gcc/Optimize-Options.html) for what `-ffast-math` actually turns on (`-funsafe-math-optimizations`, `-fno-signed-zeros`, `-fno-trapping-math`, ...). Every one of those is a specific IEEE guarantee you're giving up.


In [ ]:
# [Hub] Same source, two builds, compare last-bit output.
(labDir / 'part4FastMath.c').write_text('/* part4FastMath.c\n * A summation whose answer depends on association order.\n * Compile TWICE:\n *   cc -O2                     -o part4Safe part4FastMath.c\n *   cc -O3 -ffast-math -march=native -o part4Fast part4FastMath.c\n * Run both. Compare last-bit output.\n * -ffast-math tells the compiler "you may reassociate FP ops for speed".\n * That reassociation changes the answer, sometimes a little, sometimes a lot.\n */\n#include <stdio.h>\nint main(void) {\n    const int N = 10000000;\n    double s = 0.0;\n    for (int i = 1; i <= N; ++i) s += 1.0 / (double)i;     /* harmonic partial sum */\n    printf("sum(1/i) for i=1..%d = %.17e\\n", N, s);\n    return 0;\n}\n')
# Standard build
runShell(f'cd {labDir} && cc -O2 -Wall -o part4Safe part4FastMath.c')
# Fast-math build
runShell(f'cd {labDir} && cc -O3 -ffast-math -march=native -o part4Fast part4FastMath.c')
safeOut, _ = runShell(f'{labDir}/part4Safe')
fastOut, _ = runShell(f'{labDir}/part4Fast')
print('--- standard -O2 ---')
print(safeOut)
print('--- -O3 -ffast-math -march=native ---')
print(fastOut)


### 🖊️ Read the two lines

They might match exactly on your machine (small reductions; the compiler didn't vectorize this one), or they might differ starting in the ~15th significant digit. **Either way, they are not guaranteed to match.** Try `--O3 -ffast-math` on a big MPI-reduced aggregate in lab 05 and you will see a difference. When you write a paper, you cite the compiler *and* the flags you used, precisely because of this.


In [ ]:
checkpoint("Part 4 - fast-math changes what compilers can do", [
    check("part4Safe built", fileExists(str(labDir / 'part4Safe'))),
    check("part4Fast built", fileExists(str(labDir / 'part4Fast'))),
])


## Part 5 · Timer resolution · which clock to use

Every HPC timing you produce this semester should use `clock_gettime(CLOCK_MONOTONIC, ...)`. Here is why the alternatives are wrong:

| Function | Reports | Resolution | Good for |
|---|---|---|---|
| `time()` | wall time in whole seconds | 1 second | logging events, not benchmarks |
| `clock()` | CPU time (userspace) | ~10 ms | measuring CPU cost, NOT wall time |
| `gettimeofday()` | wall time | 1 microsecond | legacy; use `clock_gettime` instead |
| `clock_gettime(CLOCK_MONOTONIC, ...)` | wall time, never goes backwards | ~1 nanosecond | **HPC benchmarks** |
| `clock_gettime(CLOCK_REALTIME, ...)` | wall time, can jump on NTP sync | ~1 ns | timestamps, NOT benchmarks |
| `rdtsc` (x86) | CPU cycle counter | 1 cycle | microbenchmarks; needs frequency to convert |

`CLOCK_MONOTONIC` is the right default: it counts elapsed time from a fixed reference and cannot be reset, so an admin's `date -s` command in the middle of your run cannot give you a negative elapsed time.


In [ ]:
# [Hub] Three timers, one 250ms sleep. See which one lies.
(labDir / 'part5Timers.c').write_text('/* part5Timers.c\n * Compare three ways to measure elapsed time.\n * Compile: cc -O2 -Wall -o part5 part5Timers.c\n */\n#include <stdio.h>\n#include <time.h>\n#include <unistd.h>\n\nint main(void) {\n    struct timespec tsA, tsB;\n    clock_t         cA,  cB;\n    time_t          tA,  tB;\n\n    time(&tA); cA = clock(); clock_gettime(CLOCK_MONOTONIC, &tsA);\n    /* Do something that takes ~250 ms wall time but almost no CPU time. */\n    usleep(250 * 1000);\n    time(&tB); cB = clock(); clock_gettime(CLOCK_MONOTONIC, &tsB);\n\n    double wall = (tsB.tv_sec - tsA.tv_sec) + (tsB.tv_nsec - tsA.tv_nsec) * 1e-9;\n    double cpu  = (double)(cB - cA) / (double)CLOCKS_PER_SEC;\n    double t1s  = (double)(tB - tA);\n\n    printf("time()            elapsed:  %.3f s   (integer seconds only)\\n", t1s);\n    printf("clock()           elapsed:  %.6f s   (CPU time - close to zero for a sleep!)\\n", cpu);\n    printf("clock_gettime()   elapsed:  %.6f s   (wall time, nanosecond resolution)\\n", wall);\n    printf("\\nUse clock_gettime(CLOCK_MONOTONIC, ...) for HPC timing.\\n");\n    printf("time() has 1s granularity. clock() measures CPU work, not wall time.\\n");\n    return 0;\n}\n')
buildOut, buildRc = runShell(f'cd {labDir} && cc -O2 -Wall -o part5 part5Timers.c')
if buildRc != 0: print(buildOut)
else: print(runShell(f'{labDir}/part5')[0])


### The lab 01 timer you wrote already uses `clock_gettime(CLOCK_MONOTONIC, ...)`

Look at `wallSeconds()` at the top of `heat2D.c`. That's the pattern for the rest of the semester — wrap it in a helper, call the helper at the boundaries of the interval you care about, subtract. In labs 03 (OpenMP) and 05 (MPI), you'll wrap the same call inside an OpenMP barrier or `MPI_Barrier` so the elapsed time measures the whole cohort, not just rank 0.


In [ ]:
checkpoint("Part 5 - picked a real timer", [
    check("part5 binary built", fileExists(str(labDir / 'part5'))),
])


## Part 6 · Reproducibility across parallel runs

When you sum a million doubles in parallel, each thread computes a partial sum, then a reduction combines them. The **order** in which the partial sums are combined depends on the thread count, the reduction algorithm (tree vs linear), and even NUMA layout. Because floating-point addition is not associative, **the last few bits of the answer change with N**.

Small demo: sum $\sum_{i=1}^{N} 1/i^2$ (which converges to $\pi^2/6$) forwards and backwards. Different order, different last bits. Neither is "wrong."


In [ ]:
# [Hub] Order matters. Reverse-order sums are usually closer to the true value.
(labDir / 'part6ReductionOrder.c').write_text('/* part6ReductionOrder.c\n * The same sum, computed in two orders, gives different last-bit answers.\n * That\'s not a bug - it\'s floating-point non-associativity. Parallel reductions\n * combine partial sums in an order that depends on the thread count, so\n * repeatable "same bits every run" requires effort.\n * Compile: cc -O2 -Wall -o part6 part6ReductionOrder.c\n */\n#include <stdio.h>\n#include <math.h>\nint main(void) {\n    const int N = 100000;\n    double forward = 0.0, reverse = 0.0;\n    for (int i = 1; i <= N; ++i) forward += 1.0 / (double)(i * i);\n    for (int i = N; i >= 1; --i) reverse += 1.0 / (double)(i * i);\n    printf("forward sum:   %.17e\\n", forward);\n    printf("reverse sum:   %.17e\\n", reverse);\n    printf("diff:          %+.3e   (small, but NOT zero)\\n", forward - reverse);\n    printf("target pi^2/6: %.17e\\n", M_PI * M_PI / 6.0);\n    printf("\\nThe reverse sum is closer to the true value - smaller partials go in first,\\n");\n    printf("so they aren\'t lost to roundoff when added to the running total.\\n");\n    return 0;\n}\n')
runShell(f'cd {labDir} && cc -O2 -Wall -o part6 part6ReductionOrder.c')
print(runShell(f'{labDir}/part6')[0])


### When does it matter?

- **For most scientific results** — it doesn't. You report a number to 4 significant figures, publication done. The 15th-digit noise is invisible.
- **For regression tests** — a lot. Test with tolerances (Part 2), not `==`.
- **For validation across machines / thread counts** — matters. Papers that say "same answer at 1, 4, 64 ranks" have deliberately chosen a *deterministic* reduction (ordered tree reduction, sorted partial sums, or Kahan summation on a global accumulator).
- **For chaotic systems** — enormous. A weather model with a 1e-15 initial perturbation diverges in a week of simulated time. This is why weather-forecast reproducibility is such a big deal.


In [ ]:
checkpoint("Part 6 - reduction order changes last bits", [
    check("part6 binary built", fileExists(str(labDir / 'part6'))),
])


## Wrap up

Six pitfalls, six tiny C programs that show them on your hardware. From here on:

- Every lab that introduces new math will **re-derive it inline** in its Part 1, so you never need to leave a lab mid-flow. This lab is the place all of it lives together — come back to Part N when Part N bites you.
- **Lab 02** (up next, PerformanceMeasurement) leans directly on Part 4 (fast-math) and Part 5 (timers), and adds power measurement to the mix.
- **Lab 06** (MPI Domain Decomposition) is where Part 2 (equality) and Part 6 (reduction order) will bite hardest — the halo-exchange verification test is a ULP comparison, not an `==`.

You don't have to memorize this lab. You just have to know it's here.


### Lab scorecard


In [ ]:
labSummary("Numerics for HPC")


---
### One-minute feedback

What worked, what didn't, what should be clearer. Anonymous to your classmates; goes straight to the instructor.


In [ ]:
feedback("Numerics for HPC")
